# Job Agent: Ergebnisse und Qualität

Liest die lokale Datenbank `daten/jobsuche.db` und ändert nichts an den Daten. Wenn dieses Notebook mit Ausgaben gespeichert wird, stehen darin Firmen und Bewertungen.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from jobsuche import analyse, config, db

conn = db.verbinden()
pd.set_option('display.max_colwidth', 80)

FARBE = '#2a78d6'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
                     'grid.color': '#e4e4e4', 'grid.linewidth': 0.8, 'figure.dpi': 110})

## 1. Trichter: Wo bleiben die Firmen hängen?

In [ ]:
t = analyse.trichter(conn)
fig, ax = plt.subplots(figsize=(7, 3))
balken = ax.barh(t['Schritt'], t['Anzahl'], color=FARBE, height=0.55)
ax.bar_label(balken, padding=4)
ax.invert_yaxis()
ax.grid(axis='y', visible=False)
ax.set_xlabel('Firmen')
ax.set_title('Firmen je Schritt', loc='left')
plt.tight_layout()
plt.show()

## 2. Suchqualität: Findet das RAG den richtigen Fakt?

Testfragen stehen in `daten/suchfragen.json`. Gemessen wird über alle Fakten, auch unbestätigte. Vergleich: Vektor, Stichwort und Hybrid.

- **trefferquote:** richtiger Fakt unter den ersten 5
- **treffer_auf_1:** richtiger Fakt auf Platz 1
- **mrr:** mittlerer Kehrwert des Rangs; 1,0 wäre perfekt

In [ ]:
pfad = config.DATEN / 'suchfragen.json'
if not pfad.exists():
    print('Keine Testfragen: daten/suchfragen.json anlegen, Format [{"frage": ..., "erwartet": ...}].')
else:
    uebersicht, details = analyse.suchqualitaet(conn, analyse.lade_fragen(pfad), k=5)
    display(uebersicht)
    print('Fragen, bei denen der Hybrid-Modus nicht auf Platz 1 liegt:')
    display(details[(details['modus'] == 'hybrid') & (details['rang'] != 1)][['frage', 'rang']])

### Material pro Firma

Ohne Firmentext beruht ein Score nur auf der Notiz aus der Recherche.

In [ ]:
analyse.firmenabdeckung(conn)

## 3. Bewertung: Wie nah liegt die KI an deinem Urteil?

Vergleich mit deiner Bewertung unter „Firmen“. Mit wenigen Paaren ist das nur ein Hinweis.

In [ ]:
print(analyse.bewertungsguete(conn))
analyse.bewertungen(conn)

## 4. Lernen: Musst du weniger korrigieren?

Anteil der Wörter, die du am KI-Entwurf geändert hast, in der Reihenfolge der Entwürfe. Sinkt die Linie, greift das Lernen.

In [ ]:
k = analyse.korrekturgrad(conn)
if k.empty:
    print('Noch keine Entwürfe.')
else:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(1, len(k) + 1), k['aenderung_prozent'], color=FARBE, linewidth=2, marker='o', markersize=5)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Entwurf Nr.')
    ax.set_ylabel('Geändert in %')
    ax.set_title('Korrekturgrad je Entwurf', loc='left')
    plt.tight_layout()
    plt.show()
    display(k)

## 5. Ergebnis: Sagt der Score etwas über Antworten aus?

Das ist der eigentliche Test: Antworten die hoch bewerteten Firmen häufiger? Unter etwa 20 gesendeten Mails ist das noch keine belastbare Aussage.

In [ ]:
analyse.ergebnis_nach_score(conn)